# Experiment 6: Sequence Prediction using LSTM and SimpleRNN
**Student Roll Number:** `CH.SC.U4AIE24015`  



### Stock Prediction & IMDB Sentiment with RNN/LSTM


In [1]:
# Experiment 6: Sequence Prediction using LSTM and RNN
# File: Exp_06_Sequence_Prediction_RNN_LSTM.ipynb
# Student Roll Number: CH.SC.U4AIE24015

# Install yfinance if not present
!pip install -q yfinance

import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("=" * 60)
print("Student Roll Number: CH.SC.U4AIE24015")
print("Experiment 6: Sequence Prediction (Stock SimpleRNN & IMDB LSTM)")
print("=" * 60)

# PART 1: Stock Price Prediction using SimpleRNN
print("\n>>> Part 1: Stock Price Prediction with SimpleRNN")
stock = yf.download("AAPL", start="2022-01-01", end="2023-12-31", progress=False)
data = stock[["Close"]].values

scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

# Window sequences
def create_dataset(dataset, time_step=10):
    X, y = [], []
    for i in range(len(dataset) - time_step):
        X.append(dataset[i:(i + time_step), 0])
        y.append(dataset[i + time_step, 0])
    return np.array(X), np.array(y)

time_step = 10
X_seq, y_seq = create_dataset(scaled_data, time_step)
train_size = int(len(X_seq) * 0.8)
X_train, X_test = X_seq[:train_size], X_seq[train_size:]
y_train, y_test = y_seq[:train_size], y_seq[train_size:]

X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

rnn_stock = keras.Sequential([
    layers.Input(shape=(time_step, 1)),
    layers.SimpleRNN(32, activation="relu"),
    layers.Dense(1)
])
rnn_stock.compile(optimizer="adam", loss="mse")
rnn_stock.fit(X_train, y_train, epochs=20, batch_size=16, verbose=0)

y_pred_stock = rnn_stock.predict(X_test, verbose=0)
mse = mean_squared_error(y_test, y_pred_stock)
mae = mean_absolute_error(y_test, y_pred_stock)
print(f"[Roll No: CH.SC.U4AIE24015] Stock SimpleRNN -> MSE: {mse:.5f}, MAE: {mae:.5f}")

# PART 2 & 3: IMDB Sentiment Analysis with SimpleRNN vs LSTM
print("\n>>> Part 2 & 3: IMDB Sentiment Classification (SimpleRNN vs LSTM)")
max_features = 5000
maxlen = 100

(X_tr_imdb, y_tr_imdb), (X_te_imdb, y_te_imdb) = keras.datasets.imdb.load_data(num_words=max_features)
X_tr_imdb = keras.preprocessing.sequence.pad_sequences(X_tr_imdb, maxlen=maxlen)
X_te_imdb = keras.preprocessing.sequence.pad_sequences(X_te_imdb, maxlen=maxlen)

# Subset for fast lab demonstration
X_tr_sub, y_tr_sub = X_tr_imdb[:3000], y_tr_imdb[:3000]
X_te_sub, y_te_sub = X_te_imdb[:1000], y_te_imdb[:1000]

# SimpleRNN Model
rnn_imdb = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(max_features, 32),
    layers.SimpleRNN(32),
    layers.Dense(1, activation="sigmoid")
])
rnn_imdb.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
rnn_imdb.fit(X_tr_sub, y_tr_sub, epochs=3, batch_size=64, verbose=0)
_, rnn_acc = rnn_imdb.evaluate(X_te_sub, y_te_sub, verbose=0)
print(f"[Roll No: CH.SC.U4AIE24015] IMDB SimpleRNN Test Accuracy: {rnn_acc*100:.2f}%")

# LSTM Model
lstm_imdb = keras.Sequential([
    layers.Input(shape=(maxlen,)),
    layers.Embedding(max_features, 32),
    layers.LSTM(32),
    layers.Dense(1, activation="sigmoid")
])
lstm_imdb.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_imdb.fit(X_tr_sub, y_tr_sub, epochs=3, batch_size=64, verbose=0)
_, lstm_acc = lstm_imdb.evaluate(X_te_sub, y_te_sub, verbose=0)
print(f"[Roll No: CH.SC.U4AIE24015] IMDB LSTM Test Accuracy:      {lstm_acc*100:.2f}%")

print("\n" + "=" * 60)
print(f"Sequence Models Comparison Summary [Roll No: CH.SC.U4AIE24015]:")
print(f"SimpleRNN IMDB Accuracy: {rnn_acc*100:.2f}% | LSTM IMDB Accuracy: {lstm_acc*100:.2f}%")
print("=" * 60)


Student Roll Number: CH.SC.U4AIE24015
Experiment 6: Sequence Prediction (Stock SimpleRNN & IMDB LSTM)

>>> Part 1: Stock Price Prediction with SimpleRNN


/tmp/ipykernel_1651/2958584859.py:24: FutureWarning: YF.download() has changed argument auto_adjust default to True
  stock = yf.download("AAPL", start="2022-01-01", end="2023-12-31", progress=False)


[Roll No: CH.SC.U4AIE24015] Stock SimpleRNN -> MSE: 0.00101, MAE: 0.02423

>>> Part 2 & 3: IMDB Sentiment Classification (SimpleRNN vs LSTM)
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
[Roll No: CH.SC.U4AIE24015] IMDB SimpleRNN Test Accuracy: 54.80%
[Roll No: CH.SC.U4AIE24015] IMDB LSTM Test Accuracy:      79.60%

Sequence Models Comparison Summary [Roll No: CH.SC.U4AIE24015]:
SimpleRNN IMDB Accuracy: 54.80% | LSTM IMDB Accuracy: 79.60%
